In [ ]:
"""Compara detectores sobre fotos del robot etiquetadas a mano, sin depender de ultralytics.
Las clases se comparan por NOMBRE (v8 tiene 11 clases y v9 tiene 12). col_sana no cuenta como plaga."""
import math

NO_PLAGA = {"col_sana"}


def iou(a, b):
    """Cajas (x, y, w, h) con centro normalizado."""
    ix = min(a[0] + a[2] / 2, b[0] + b[2] / 2) - max(a[0] - a[2] / 2, b[0] - b[2] / 2)
    iy = min(a[1] + a[3] / 2, b[1] + b[3] / 2) - max(a[1] - a[3] / 2, b[1] - b[3] / 2)
    if ix <= 0 or iy <= 0:
        return 0.0
    inter = ix * iy
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)


def wilson(k, n, z=1.96):
    """Intervalo de confianza del 95 % para k aciertos en n casos; None si no hay casos."""
    if n == 0:
        return None
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return round((c - m) / d, 3), round((c + m) / d, 3)


def _razon(k, n):
    return {"valor": round(k / n, 3) if n else None, "k": k, "n": n, "ic95": wilson(k, n)}


def evaluar(fotos, iou_min=0.3):
    """fotos: lista de {"real": [(clase, x, y, w, h)], "pred": [(clase, conf, x, y, w, h)]}.
    Cada caja predicha empareja como máximo una real. Devuelve exhaustividad y precisión por caja, y resultados por imagen."""
    tp_any = tp_clase = n_real = n_pred = 0
    img_con, img_con_ok, img_sin, img_sin_aviso = 0, 0, 0, 0
    por_clase = {}
    for f in fotos:
        real = [r for r in f["real"] if r[0] not in NO_PLAGA]
        pred = sorted((p for p in f["pred"] if p[0] not in NO_PLAGA), key=lambda p: -p[1])
        n_real += len(real); n_pred += len(pred)
        libres = list(range(len(real)))
        for p in pred:
            candidatos = sorted(((iou(p[2:], real[i][1:]), i) for i in libres), reverse=True)
            if candidatos and candidatos[0][0] >= iou_min:
                # Prefiere una caja real de la misma clase entre las que solapan lo suficiente.
                i = next((i for v, i in candidatos if v >= iou_min and real[i][0] == p[0]), candidatos[0][1])
                libres.remove(i); tp_any += 1
                tp_clase += real[i][0] == p[0]
                por_clase.setdefault(real[i][0], [0, 0])[0] += real[i][0] == p[0]
        for r in real:
            por_clase.setdefault(r[0], [0, 0])[1] += 1
        if real:
            img_con += 1; img_con_ok += bool(pred)
        else:
            img_sin += 1; img_sin_aviso += bool(pred)
    return {"exhaustividad_caja_cualquier_clase": _razon(tp_any, n_real), "exhaustividad_caja_misma_clase": _razon(tp_clase, n_real),
            "precision_caja": _razon(tp_any, n_pred), "imagenes_con_plaga_avisadas": _razon(img_con_ok, img_con),
            "imagenes_sin_plaga_con_falso_aviso": _razon(img_sin_aviso, img_sin),
            "por_clase": {c: _razon(k, n) for c, (k, n) in sorted(por_clase.items())}}


def leer_etiqueta(texto, nombres):
    return [(nombres[int(p[0])], *map(float, p[1:5])) for p in (l.split() for l in texto.splitlines()) if len(p) == 5]


# PLAGASX AJUSTE (2026-11-09): reajusta v9 con cajas a mano sobre fotos del robot y compara v8, v9 y el reajuste. Generado por crear_notebook.py.
# PLAGASX v9 (2026-11-08): coles y coliflores, 12 clases (añade col_trips), yolo11m, 960 px, 2 GPU T4.
# AGROBOT · PLAGASX v6 (negativos de interior + cajas reales de daño de hoja, mosca blanca, pulgón, orugas y pulguilla) — base v2 — coles: plagas y enfermedades. Combina: (1) tu dataset de coles, (2) tres datasets externos de coliflor/papa,
# (3) tu dataset de SENDEROS ('navegacion') como imágenes de fondo (plantas y caminos SIN enfermedad: reducen falsos positivos).
# Entrena (a) un DETECTOR YOLO y (b) un CLASIFICADOR YOLO-cls. Las etiquetas de los datasets externos son de imagen completa, así que las
# métricas que valen son las del CLASIFICADOR; el detector sirve para localizar la planta. Todo se imprime con tablas y gráficas.
import os, sys, json, random, shutil, subprocess, re, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import cv2
try:
    from IPython.display import display
except Exception:
    display = print

DRY = bool(os.environ.get("PLAGAS_DRY"))                       # solo arma el dataset (prueba local, sin entrenar)
ENTRADA = Path(os.environ.get("PLAGAS_INPUT", "/kaggle/input"))
SALIDA = Path(os.environ.get("PLAGAS_SALIDA", "/kaggle/working")) / "dataset_combinado"
RES = SALIDA.parent
random.seed(42); np.random.seed(42)
if not DRY:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)

CLASES = ["col_sana", "col_amarillamiento", "col_manchas_cafe", "col_dano_insecto", "col_mildiu_velloso", "col_malformacion", "col_pudricion", "col_mosca_blanca", "col_pulgon", "col_oruga", "col_pulguilla", "col_trips"]
# nombre de carpeta externo (normalizado: minúsculas y solo letras/números) -> clase
MAPA = {"healthy": 0, "nodisease": 0, "blackrot": 2, "alternarialeafspot": 2, "alternaria": 2, "bacterialspot": 2, "bacterialspotrot": 2,
        "ringspot": 2, "leafspot": 2, "downymildew": 4, "softrot": 6, "clubroot": 5,
        "cabbageaphidcolony": 3, "greenpeachaphid": 3, "potatoleafminer": 3, "leafminer": 3, "aphid": 3}
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
MAX_POR_CLASE, MAX_FONDO = 500, 120

# ---------------- localizar los datasets montados (Kaggle cambia la ruta según la versión) ----------------
def buscar_dir(pred, base=ENTRADA):
    for p in sorted(base.rglob("*")):
        if p.is_dir() and pred(p):
            return p
    return None
mi = buscar_dir(lambda p: (p / "images").is_dir() and (p / "labels").is_dir() and (p / "dataset.yaml").exists()) \
     or buscar_dir(lambda p: (p / "images").is_dir() and (p / "labels").is_dir() and "coles" in str(p).lower())
nav = buscar_dir(lambda p: p.is_dir() and (p / "images").is_dir() and any(p.joinpath("images").glob("camino-*.jpg")))
neg = buscar_dir(lambda p: p.is_dir() and "negativos" in p.name.lower() and (p / "images").is_dir())
dir_cajas = buscar_dir(lambda p: p.is_dir() and "cajas-plagas" in p.name.lower() and (p / "images" / "train").is_dir())
print("cajas reales (daño de hoja + mosca blanca):", dir_cajas)
ip102 = buscar_dir(lambda p: (p / "data.yaml").exists() and "蚜虫" in (p / "data.yaml").read_text(encoding="utf-8", errors="ignore"))
print("IP102 (pulgón, mosca blanca, orugas, pulguilla):", ip102)
print("tu dataset de coles :", mi); print("navegación          :", nav); print("negativos de fondo  :", neg)
print("contenido de", ENTRADA, ":", [p.name for p in ENTRADA.iterdir()][:20] if ENTRADA.exists() else "no existe")

# ---------------- calidad de imagen ----------------
def apta(img, umbral_nitidez=60.0, umbral_sujeto=0.03):
    if img is None or min(img.shape[:2]) < 64: return False
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    nit = cv2.Laplacian(gris, cv2.CV_64F).var()
    _, m = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return nit >= umbral_nitidez and np.count_nonzero(m) / m.size >= umbral_sujeto

def leer_redim(ruta, lado=960):
    img = cv2.imread(str(ruta))
    if img is None: return None
    h, w = img.shape[:2]; k = lado / max(h, w)
    return cv2.resize(img, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA) if k < 1 else img

for sp in ("train", "val", "test"):
    (SALIDA / "images" / sp).mkdir(parents=True, exist_ok=True); (SALIDA / "labels" / sp).mkdir(parents=True, exist_ok=True)
registro = []                                              # (split, origen, clase o -1 (fondo), archivo)

def guardar(img, split, nombre, cajas):
    cv2.imwrite(str(SALIDA / "images" / split / f"{nombre}.jpg"), img)
    (SALIDA / "labels" / split / f"{nombre}.txt").write_text("".join(f"{c} {x} {y} {w} {h}\n" for c, x, y, w, h in cajas))

# ---------------- 1) TU DATASET (cajas automáticas por color: ver el análisis) ----------------
if mi is not None:
    for sp in ("train", "val", "test"):
        for f in sorted((mi / "images" / sp).glob("*.jpg")):
            lbl = mi / "labels" / sp / (f.stem + ".txt")
            cajas = [l.split() for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
            if not cajas: continue
            img = leer_redim(f)
            if img is None: continue
            cajas = [(int(c[0]), *map(float, c[1:5])) for c in cajas]
            guardar(img, sp, "mio_" + f.stem, cajas)
            dom = Counter(c[0] for c in cajas).most_common(1)[0][0]
            registro.append((sp, "propio", dom, "mio_" + f.stem))
print("imágenes propias incluidas:", sum(1 for r in registro if r[1] == "propio"))

# ---------------- 2) EXTERNOS: se recorren todas las carpetas de /kaggle/input que no sean las propias ----------------
externas, no_mapeadas = [], Counter()
excluir = [p for p in (mi, nav) if p is not None]
for raiz, dirs, archivos in os.walk(ENTRADA, followlinks=True):
    rp = Path(raiz)
    if any(str(rp).startswith(str(e)) for e in excluir) or "agrobot-slam" in str(rp): continue
    imgs = [a for a in archivos if a.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not imgs: continue
    k = MAPA.get(norm(rp.name))
    if k is None:
        no_mapeadas[rp.name] += len(imgs); continue
    externas += [(rp / a, k) for a in imgs]
print("carpetas externas sin mapear (se ignoran):", dict(no_mapeadas.most_common(12)))
por_clase = defaultdict(list)
for r, k in externas: por_clase[k].append(r)
sel = []
for k, rs in por_clase.items():
    random.shuffle(rs); n = 0
    for r in rs:
        if n >= MAX_POR_CLASE: break
        img = leer_redim(r)
        if apta(img): sel.append((img, k)); n += 1
random.shuffle(sel)
corte = [int(len(sel) * .7), int(len(sel) * .9)]
for i, (img, k) in enumerate(sel):
    sp = "train" if i < corte[0] else ("val" if i < corte[1] else "test")
    guardar(img, sp, f"ext_{i:05d}", [(k, .5, .5, 1.0, 1.0)])                 # imagen completa (clasificación, no detección real)
    registro.append((sp, "externo", k, f"ext_{i:05d}"))
print("imágenes externas incluidas:", len(sel))

# ---------------- 3) TU DATASET DE SENDEROS COMO FONDO (etiqueta vacía = «aquí no hay enfermedad») ----------------
if nav is not None:
    fotos = sorted((nav / "images").glob("camino-*.jpg"))
    paso = max(1, len(fotos) // MAX_FONDO)
    for j, f in enumerate(fotos[::paso][:MAX_FONDO]):
        img = leer_redim(f)
        if img is None: continue
        sp = "val" if j % 5 == 0 else "train"
        guardar(img, sp, f"fondo_{j:04d}", [])
        registro.append((sp, "fondo", -1, f"fondo_{j:04d}"))
print("imágenes de fondo (senderos):", sum(1 for r in registro if r[1] == "fondo"))

# ---------------- 3b) CAJAS REALES (v5, 2026-10-22): hojas de col dañadas/comidas (clase 3) y mosca blanca (clase 7) ----------------
# Cajas dibujadas a mano sobre la imagen (no de imagen completa): mejoran la localización. Las de mosca blanca son fotos de YUCA (fondo
# distinto a la col); las de daño, de col blanca (CC-BY-4.0, Arko007/white-cabbage-leaf-damage). Etiqueta vacía = hoja sin daño.
if dir_cajas is not None:
    for sp in ("train", "val", "test"):
        for f in sorted((dir_cajas / "images" / sp).glob("*.jpg")):
            lab = dir_cajas / "labels" / sp / (f.stem + ".txt")
            img = cv2.imread(str(f))
            if img is None: continue
            filas = [l.split() for l in lab.read_text().splitlines() if len(l.split()) >= 5] if lab.exists() else []
            guardar(img, sp, f"caj_{f.stem}", [(int(c[0]), c[1], c[2], c[3], c[4]) for c in filas])
            registro.append((sp, "cajas", -2, f"caj_{f.stem}"))
print("imágenes con cajas reales:", sum(1 for r in registro if r[1] == "cajas"))

# ---------------- 3c) PLAGAS DE LA COL DEL IP102 (v6, 2026-10-22) con cajas reales ----------------
# En las fotos del operador hay una colonia de PULGÓN CEROSO (Brevicoryne brassicae) y ningún modelo tenía clase de pulgón. Del IP102
# (Wu et al., CVPR 2019) se toman solo las especies que atacan coles o se le parecen, y se agrupan: pulgones (8), mosca blanca de
# invernadero (7), orugas/gusanos cortadores (9) y pulguilla/escarabajo pulga (10). Las fotos del IP102 son de internet, de cerca.
IP_MAPA = {"蚜虫": 8, "长角麦蚜": 8, "麦二叉蚜": 8, "禾谷缢管蚜": 8, "苜蓿斑蚜": 8, "橘蚜": 8, "小桔蚜": 8, "苹果绣线菊蚜": 8,
           "温室白粉虱": 7,
           "甘蓝夜蛾": 9, "东方菜粉蝶": 9, "夜蛾": 9, "甜菜夜蛾": 9, "小地老虎": 9, "大地老虎": 9, "黄地老虎": 9, "棉斜纹夜蛾": 9,
           "跳甲": 10,
           "谷物蓟马": 11, "麦管蓟马": 11, "牛角花齿蓟马": 11, "蓟马/牧草虫": 11, "芒果茶黄蓟马": 11}  # v9: trips; ninguno es de coles, medir aparte
MAX_IP_TRAIN, MAX_IP_EVAL = 1200, 200
if ip102 is not None:
    import yaml
    nombres = [str(n).strip() for n in yaml.safe_load((ip102 / "data.yaml").read_text(encoding="utf-8"))["names"]]
    ip_id = {i: IP_MAPA[n] for i, n in enumerate(nombres) if n in IP_MAPA}
    print("clases IP102 usadas:", {nombres[i]: CLASES[k] for i, k in ip_id.items()})
    for sp_src, destinos in (("train", ("train",)), ("valid", ("val", "test"))):
        dimg = next((d for d in (ip102 / sp_src / "images", ip102 / sp_src) if d.is_dir()), None)
        if dimg is None: print("IP102: falta", sp_src); continue
        dlab = dimg.parent / "labels" if dimg.name == "images" else dimg
        por_clase = defaultdict(list)
        for lab in sorted(dlab.glob("*.txt")):
            filas = [l.split() for l in lab.read_text().splitlines() if len(l.split()) >= 5]
            mias = [(ip_id[int(c[0])], c[1], c[2], c[3], c[4]) for c in filas if int(c[0]) in ip_id]
            if mias: por_clase[mias[0][0]].append((lab.stem, mias))
        tope = MAX_IP_TRAIN if sp_src == "train" else 2 * MAX_IP_EVAL
        for k, lst in por_clase.items():
            random.shuffle(lst)
            for j, (stem, mias) in enumerate(lst[:tope]):
                f = next((dimg / (stem + e) for e in (".jpg", ".jpeg", ".png", ".JPG") if (dimg / (stem + e)).exists()), None)
                img = leer_redim(f) if f is not None else None
                if img is None: continue
                sp = destinos[j % len(destinos)]
                guardar(img, sp, f"ip_{k}_{stem}", mias)
                registro.append((sp, "ip102", -2, f"ip_{k}_{stem}"))
print("imágenes IP102 incluidas:", sum(1 for r in registro if r[1] == "ip102"))

# ---------------- 4) NEGATIVOS DUROS DE INTERIOR (2026-10-22): fondos SIN planta que el detector marcó como plaga ----------------
# Pufs, puerta, parlante, zapatos y paredes que PLAGASX confundió con col_dano_insecto/col_malformacion al 77-95 % en la prueba real.
# Se usan TODOS y sin filtro de nitidez: lo borroso y oscuro de la cámara del robot es justo lo que debe aprender a ignorar.
if neg is not None:
    fotos = sorted((neg / "images").glob("*.jpg"))
    for j, f in enumerate(fotos):
        img = leer_redim(f)
        if img is None: continue
        sp = "val" if j % 5 == 0 else ("test" if j % 5 == 1 else "train")        # un test propio para medir los falsos positivos
        guardar(img, sp, f"negint_{j:04d}", [])
        registro.append((sp, "fondo_interior", -1, f"negint_{j:04d}"))
print("negativos duros de interior:", sum(1 for r in registro if r[1] == "fondo_interior"))

# ---------------- 5) FOTOS DEL ROBOT CON CAJAS A MANO: entrenan repetidas para que pesen frente a miles de imágenes externas ----------------
REPETIR = 6
raiz_op = next(p for p in sorted(ENTRADA.rglob("*")) if p.is_dir() and (p / "op_eval" / "images").is_dir() and (p / "pesos" / "v9_det.pt").exists())
for sp_op, sp, veces in (("op_train", "train", REPETIR), ("op_val", "val", 1)):
    for f in sorted((raiz_op / sp_op / "images").glob("*.jpg")):
        img = leer_redim(f)
        if img is None: continue
        cajas = [l.split() for l in (raiz_op / sp_op / "labels" / f"{f.stem}.txt").read_text().splitlines() if len(l.split()) == 5]
        for r in range(veces):
            guardar(img, sp, f"op{r}_{f.stem}", [(int(c[0]), c[1], c[2], c[3], c[4]) for c in cajas])
            registro.append((sp, "operador", -2, f"op{r}_{f.stem}"))
print("fotos del operador (con repetición):", sum(1 for r in registro if r[1] == "operador"))
assert not any(p.stem in {f.stem for f in (raiz_op / "op_train" / "images").glob("*.jpg")} for p in (raiz_op / "op_eval" / "images").glob("*.jpg"))

df = pd.DataFrame(registro, columns=["split", "origen", "clase", "archivo"])
df["clase_nombre"] = df.clase.map(lambda c: "cajas" if c == -2 else ("fondo" if c < 0 else CLASES[c]))
display(pd.crosstab(df.clase_nombre, [df.split, df.origen], margins=True))
(SALIDA / "dataset.yaml").write_text(f"path: {SALIDA}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(CLASES)}\nnames:\n" +
                                     "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASES)))
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
pd.crosstab(df.clase_nombre, df.origen).plot.bar(stacked=True, ax=axs[0]); axs[0].set_title("imágenes por clase y origen"); axs[0].grid(alpha=.3)
pd.crosstab(df.split, df.origen).plot.bar(stacked=True, ax=axs[1]); axs[1].set_title("imágenes por partición"); axs[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(RES / "01_composicion.png", dpi=110); plt.show()
if DRY: raise SystemExit("PLAGAS_DRY: dataset armado, sin entrenar")

# ---------------- 6) REAJUSTE DE v9: congela la base de la red para conservar lo aprendido ----------------
from ultralytics import YOLO
CONF = 0.45
det = YOLO(str(raiz_op / "pesos" / "v9_det.pt"))
det.train(data=str(SALIDA / "dataset.yaml"), epochs=80, time=3.0, imgsz=960, batch=16, patience=25, close_mosaic=10, device=[0, 1], project=str(RES / "runs"), name="ajuste",
          freeze=10, optimizer="AdamW", lr0=0.0005, lrf=0.1, mosaic=0.5, mixup=0.0, hsv_h=.015, hsv_s=.5, hsv_v=.4, fliplr=.5, flipud=0.0, exist_ok=True, verbose=False)
shutil.copy(RES / "runs" / "ajuste" / "weights" / "best.pt", RES / "coles_det_ajuste.pt")

# ---------------- 7) COMPARACIÓN EN FOTOS DEL ROBOT QUE NINGÚN REAJUSTE VIO ----------------
fotos_eval = sorted((raiz_op / "op_eval" / "images").glob("*.jpg"))
def predecir(ruta):
    m = YOLO(str(ruta)); salida = []
    for f in fotos_eval:
        r = m.predict(str(f), imgsz=960, conf=CONF, verbose=False, device=0)[0]
        pred = [(r.names[int(c)], float(p), *map(float, b)) for c, p, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xywhn)]
        salida.append({"real": leer_etiqueta((raiz_op / "op_eval" / "labels" / f"{f.stem}.txt").read_text(), CLASES), "pred": pred})
    return salida
modelos = {"v8": raiz_op / "pesos" / "v8_det.pt", "v9": raiz_op / "pesos" / "v9_det.pt", "ajuste": RES / "coles_det_ajuste.pt"}
resultado = {n: evaluar(predecir(p)) for n, p in modelos.items()}
(RES / "comparacion_robot.json").write_text(json.dumps({"fotos_evaluacion": len(fotos_eval), "conf": CONF, "modelos": resultado}, ensure_ascii=False, indent=1))
display(pd.DataFrame({n: {k: f"{v['valor']} ({v['k']}/{v['n']}) IC95 {v['ic95']}" for k, v in r.items() if k != "por_clase"} for n, r in resultado.items()}))
print("HECHO. coles_det_ajuste.pt y comparacion_robot.json. Se conservan las fotos de evaluación fuera del entrenamiento; con pocas fotos, mirar los intervalos antes de elegir modelo.")
